In [2]:
from collections import deque
import random
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

In [3]:
# Separate function/class to build the Q-Network architecture
def build_q_network(state_dim=2, action_dim=3, hidden_dim=64):
  return nn.Sequential(
      nn.Linear(state_dim, hidden_dim),
      nn.ReLU(),
      nn.Linear(hidden_dim, hidden_dim),
      nn.ReLU(),
      nn.Linear(hidden_dim, action_dim),
  )

In [4]:
# Experience Replay Buffer using python collections.deque
class ReplayBuffer:

  def __init__(self, capacity=10000):
    self.buffer = deque(maxlen=capacity)

  def push(self, state, action, reward, next_state, done):
    self.buffer.append((state, action, reward, next_state, done))

  def sample(self, batch_size):
    batch = random.sample(self.buffer, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    return (
        torch.tensor(states, dtype=torch.float32),
        torch.tensor(actions, dtype=torch.int64),
        torch.tensor(rewards, dtype=torch.float32),
        torch.tensor(next_states, dtype=torch.float32),
        torch.tensor(dones, dtype=torch.float32),
    )

  def __len__(self):
    return len(self.buffer)

In [5]:
# Environment & Hyperparameters
env = gym.make("MountainCar-v0")
state_dim = env.observation_space.shape[0]  # 2 (position, velocity)
action_dim = env.action_space.n  # 3 discrete actions

gamma = 0.99  # Discount factor
lr = 0.001  # Learning rate
epsilon = 1.0  # Initial exploration rate
epsilon_min = 0.01  # Minimum exploration rate
epsilon_decay = 0.995  # Decay per episode
batch_size = 64  # Mini-batch size
target_update_freq = 10  # Copy target weights every N episodes

# Instantiate Networks, Memory Buffer, &amp; Optimizer
q_net = build_q_network(state_dim, action_dim)
target_net = build_q_network(state_dim, action_dim)
target_net.load_state_dict(q_net.state_dict())  # Synchronize initial weights

optimizer = optim.Adam(q_net.parameters(), lr=lr)
loss_fn = nn.MSELoss()
replay_buffer = ReplayBuffer(capacity=20000)

num_episodes = 800
goal_reached_count = 0

In [6]:
# MAIN TRAINING LOOP
for episode in range(1, num_episodes + 1):
  state, info = env.reset()  # Reset environment at episode start
  total_reward = 0
  done = False

  while not done:
    # Epsilon-Greedy Action Selection
    if random.random() < epsilon:
      action = env.action_space.sample()  # Explore randomly
    else:
      state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
      with torch.no_grad():
        q_values = q_net(state_tensor)
        action = q_values.argmax().item()  # Exploit highest predicted Q-value

    # Transition Step in Environment
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # Apply Custom Reward Shaping
    pos, vel = next_state[0], next_state[1] 
    shaped_reward = reward + (pos + 0.5) + (abs(vel) * 10.0)

    # Store Experience Tuple in Replay Buffer
    replay_buffer.push(state, action, shaped_reward, next_state, done)
    state = next_state
    total_reward += reward

    # Train Network on Mini-batch from Replay Memory
    if len(replay_buffer) >= batch_size:
      b_states, b_actions, b_rewards, b_next_states, b_dones = (
          replay_buffer.sample(batch_size)
      )

      # Current predicted Q-values: Q(s, a)
      q_eval = q_net(b_states).gather(1, b_actions.unsqueeze(1)).squeeze(1)

      # Bellman Target Calculation using Target Network: Q_target = R + gamma * max(Q(s', a'))
      with torch.no_grad():
        max_next_q = target_net(b_next_states).max(1).values
        q_target = b_rewards + (gamma * max_next_q * (1 - b_dones))

      # Compute MSE Loss and perform Gradient Descent
      loss = loss_fn(q_eval, q_target)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # Decay Epsilon after each episode
  epsilon = max(epsilon_min, epsilon * epsilon_decay)

  # Periodically update Target Network weights
  if episode % target_update_freq == 0:
    target_net.load_state_dict(q_net.state_dict())

  # Log progress and check if goal flag was reached
  if next_state[0] >= 0.5:
    goal_reached_count += 1
    print(
        f"Episode {episode} | Goal Reached | Final Position: {next_state[0]:.2f}"
    )
  elif episode % 20 == 0:
    print(
        f"Episode {episode}/{num_episodes} | Total Reward: {total_reward} |"
        f" Epsilon: {epsilon:.3f}"
    )

  # Save model state upon successful goal achievement threshold
  if goal_reached_count >= 5:
    torch.save(q_net.state_dict(), "dqn_mountaincar.pth")
    print("Model saved")
    break

env.close()

C:\Users\jayog\AppData\Local\Temp\ipykernel_22284\2496391187.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  torch.tensor(states, dtype=torch.float32),


Episode 20/800 | Total Reward: -200.0 | Epsilon: 0.905
Episode 40/800 | Total Reward: -200.0 | Epsilon: 0.818
Episode 60/800 | Total Reward: -200.0 | Epsilon: 0.740
Episode 80/800 | Total Reward: -200.0 | Epsilon: 0.670
Episode 100/800 | Total Reward: -200.0 | Epsilon: 0.606
Episode 120/800 | Total Reward: -200.0 | Epsilon: 0.548
Episode 140/800 | Total Reward: -200.0 | Epsilon: 0.496
Episode 155 | Goal Reached | Final Position: 0.51
Episode 160/800 | Total Reward: -200.0 | Epsilon: 0.448
Episode 176 | Goal Reached | Final Position: 0.52
Episode 180/800 | Total Reward: -200.0 | Epsilon: 0.406
Episode 182 | Goal Reached | Final Position: 0.51
Episode 184 | Goal Reached | Final Position: 0.52
Episode 186 | Goal Reached | Final Position: 0.51
Model saved


In [7]:
# EVALUATION LOOP
test_env = gym.make("MountainCar-v0", render_mode="human")
policy_net = build_q_network(state_dim, action_dim)
policy_net.load_state_dict(torch.load("dqn_mountaincar.pth"))
policy_net.eval()

print("\nRunning Test Episodes")
for test_ep in range(1, 6):
  state, info = test_env.reset()
  done = False
  steps = 0

  while not done:
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
      action = policy_net(state_tensor).argmax().item()

    state, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    steps += 1

  print(
      f"Test Episode {test_ep}: Reached Flag in {steps} steps. Final Position:"
      f" {state[0]:.2f}"
  )

test_env.close()


Running Test Episodes
Test Episode 1: Reached Flag in 120 steps. Final Position: 0.53
Test Episode 2: Reached Flag in 116 steps. Final Position: 0.52
Test Episode 3: Reached Flag in 114 steps. Final Position: 0.52
Test Episode 4: Reached Flag in 116 steps. Final Position: 0.50
Test Episode 5: Reached Flag in 118 steps. Final Position: 0.51
